## Setup & Imports

In this section, we import the necessary libraries to handle data loading, text processing, and mathematical operations. 

* **`datasets`**: For loading our data via the Hugging Face hub.
* **`string`, `nltk`, & `word_tokenize`**: For text cleaning, processing, and breaking strings down into tokens.
* **`Counter` & `defaultdict`**: For frequency analysis and efficiently managing dictionaries.
* **`math`**: For any necessary mathematical computations.
* **`pickle`**: For saving and loading serialized data/models later.

In [28]:
import string
from collections import Counter, defaultdict
import nltk
import pickle
import math
from datasets import load_dataset
from nltk.tokenize import word_tokenize

## NLTK Data Initialization

Before we can begin processing our text, we need to download the necessary data packages for NLTK's tokenization tools. 

* **`punkt`**: This downloads the pre-trained tokenizer model, which is essential for accurately splitting text into sentences and individual words.
* **`punkt_tab`**: This provides the supplementary data files required by the Punkt tokenizer to function correctly in the latest versions of NLTK.

In [29]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alyad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alyad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Data Loading & Preparation

In this section, we retrieve our corpus and format both our training and testing data so it is ready for processing.

* **Loading the Dataset**: We use the Hugging Face `datasets` library to fetch the raw WikiText-2 dataset (`wikitext-2-raw-v1`), which consists of a large collection of high-quality Wikipedia articles.
* **Preparing the Training Data (`train_text`)**: We extract the 'train' split of the dataset. Because the text is provided as a list of separate paragraphs, we "flatten" it by joining everything together with newline characters (`\n`). This yields one massive, continuous string of text to train our model on.
* **Preparing the Test Data (`test_text`)**: We apply the exact same flattening process to the 'test' split. We will set this string aside and use it later to evaluate how well our finished model performs on unseen data.

In [30]:
print("Loading WikiText-2 dataset...")
# fetching the raw Wikipedia corpus
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
# flattening the list of Wikipedia paragraphs into one continuous string for the training phase
train_text = "\n".join(dataset['train']['text'])
# Prepare test data 
test_text = "\n".join(dataset['test']['text'])

Loading WikiText-2 dataset...


## Text Preprocessing & Tokenization

Before we can build a language model, we need to clean our raw text and break it down into manageable pieces. The `preprocess_text` function handles this in three steps:

* **Lowercasing (`text.lower()`)**: We convert all text to lowercase so that words like "The" and "the" are recognized as the exact same word by our model.
* **Removing Punctuation (`text.translate(...)`)**: We strip out all punctuation marks (commas, periods, quotes) to focus purely on the vocabulary and sequence of words.
* **Tokenization (`word_tokenize(text)`)**: We use NLTK to chop our massive, continuous string of text into a Python list of individual words (called "tokens"). 

**What this looks like in practice:**
* **Input:** `"The quick brown fox, jumps over the lazy dog!"`
* **Output:** `['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']`

Finally, we apply this preprocessing pipeline to both our `train_text` and `test_text`. 
We also print out the total number of tokens and the first 10 words of each set as a quick sanity check to ensure our data is clean and ready for modeling.

In [31]:
def preprocess_text(text):
    # converting all characters to lowercase
    text = text.lower()
    # removing all punctuation marks to focus on word sequences
    text = text.translate(str.maketrans('', '', string.punctuation))
    # spliting the long string into a Python list of individual word tokens
    tokens = word_tokenize(text)
    return tokens

# executing the preprocessing on the raw training text
print("Preprocessing tokens...")
train_tokens = preprocess_text(train_text)
print(f"Number of tokens in training set: {len(train_tokens)}")
print(f"First 10 tokens: {train_tokens[:10]}")
test_tokens = preprocess_text(test_text)
print(f"Number of tokens in test set: {len(test_tokens)}")
print(f"First 10 tokens: {test_tokens[:10]}")

Preprocessing tokens...
Number of tokens in training set: 1756302
First 10 tokens: ['valkyria', 'chronicles', 'iii', 'senjō', 'no', 'valkyria', '3', 'unrecorded', 'chronicles', 'japanese']
Number of tokens in test set: 207059
First 10 tokens: ['robert', 'boulter', 'robert', 'boulter', 'is', 'an', 'english', 'film', 'television', 'and']


## The N-Gram Language Model Class

In this section, we define the core engine of our language model: the `NGramModel` class. This class handles processing the text, counting word sequences, and calculating the mathematical probabilities of upcoming words.

### Class Initialization (`__init__`)
When we create a new model instance, we set up several foundational pieces:
* **N-Gram Order (`n`)**: Determines the size of the window the model looks at (e.g., if `n=3`, the model uses the previous 2 words to predict the 3rd).
* **Vocabulary Tracking (`vocab` & `vocab_size`)**: Identifies every unique word in the training data, which is essential for our probability smoothing calculations later.
* **Tracking Frequencies (`ngram_counts` & `context_counts`)**: Initializes specialized dictionaries (`defaultdict` and `Counter`) to efficiently store how often specific word contexts appear and which target words follow them.

### Counting the N-Grams (`_build_model`)
To populate our frequency counts, the model utilizes a "sliding window" technique across our list of training tokens. 



As the window slides across the text one word at a time, it divides the tokens into two parts:
* **Context**: The preceding sequence of words (length of `n - 1`) that the model uses as a historical clue.
* **Target**: The single word that immediately follows the context.

For every step of the window, we update our counters to record that we saw this specific context, and that this specific target word followed it.

### Calculating Probabilities with Laplace Smoothing (`get_probability`)
This method calculates the probability of a specific word appearing after a given context. Crucially, it introduces **Laplace Smoothing** (also known as Add-One Smoothing).

Without smoothing, if we test our model on a word sequence it never saw during training, the probability drops to zero. To fix this, we mathematically "pretend" we've seen every possible word combination at least once by adding 1 to the count, and balancing the denominator with our vocabulary size.

The formula implemented in the code is:

$$P(w | c) = \frac{C(c, w) + 1}{C(c) + |V|}$$

Where:
* $P(w | c)$ is the probability of the target word ($w$) given the context ($c$).
* $C(c, w)$ is the number of times the target word immediately followed the context (`count_w_context`).
* $C(c)$ is the total number of times the context appeared in the text (`count_context`).
* $|V|$ is the total number of unique words in our training vocabulary (`self.vocab_size`).

In [33]:
class NGramModel:
    def __init__(self, n, tokens):
        # set N-gram order
        self.n = n
        # identify unique words in training data
        self.vocab = set(tokens)
        # total unique words (V) for smoothing
        self.vocab_size = len(self.vocab)
        # dictionary to store word frequencies after contexts
        self.ngram_counts = defaultdict(Counter)
        # frequency of each context appearing in text
        self.context_counts = Counter()
        # populate counts from tokens
        self._build_model(tokens)

    def _build_model(self, tokens):
        # sliding a window across the tokens to count occurrences of n-sized sequences
        for i in range(len(tokens) - self.n + 1):
            # the words the model looks at (n-1 words)
            context = tuple(tokens[i : i + self.n - 1])
            # the next word the model is trying to predict after context
            target = tokens[i + self.n - 1]

            self.ngram_counts[context][target] += 1
            self.context_counts[context] += 1
    def get_probability(self, word, context):
        # calculates the smoothed probability of a word given a context.
        # Formula: (Count of sequence + 1) / (Count of context + Vocabulary Size)
        context = tuple(context)
        count_w_context = self.ngram_counts[context][word]
        count_context = self.context_counts[context]

        # applying Laplace smoothing to avoid zero-probability errors
        probability = (count_w_context + 1) / (count_context + self.vocab_size)
        return probability



## Instantiating the N-Gram Models

With our `NGramModel` class defined, we can now build several distinct language models with varying context windows (values of *n*). 



By creating these different models, we can later compare how increasing the amount of historical context impacts the model's predictive performance:

* **Unigram Model (`n=1`)**: This model assumes words appear completely independently of one another. Its context is entirely empty `()`, meaning it bases its probabilities purely on the overall frequency of individual words.
* **Bigram Model (`n=2`)**: This model relies on exactly one preceding word to predict the next. Its context is a single word `(word1,)`.
* **Trigram Model (`n=3`)**: This model looks at the two preceding words to make its prediction. Its context is a pair of words `(word1, word2)`.
* **Quadgram Model (`n=4`)**: This model looks at a broader history of three preceding words. Its context is a sequence of three words `(word1, word2, word3)`.

In [ ]:
print("Building various N-gram models...")
# unigram (n=1): Context is empty tuple ()
unigram_model = NGramModel(n=1, tokens=train_tokens)
# bigram (n=2): Context is (word1,)
bigram_model  = NGramModel(n=2, tokens=train_tokens)
# trigram (n=3): Context is (word1, word2)
trigram_model = NGramModel(n=3, tokens=train_tokens)
# quadgram (n=4): Context is (word1, word2, word3)
quadgram_model = NGramModel(n=4, tokens=train_tokens)

print("\nModel Statistics:")
print(f"Vocabulary size: {unigram_model.vocab_size} unique words")
print(f"\nUnigram model: {sum(unigram_model.context_counts.values())} total words counted")
print(f"Bigram model: {len(bigram_model.context_counts)} unique contexts, {sum(bigram_model.context_counts.values())} total bigrams")
print(f"Trigram model: {len(trigram_model.context_counts)} unique contexts, {sum(trigram_model.context_counts.values())} total trigrams")
print(f"Quadgram model: {len(quadgram_model.context_counts)} unique contexts, {sum(quadgram_model.context_counts.values())} total quadgrams")


Building various N-gram models...

Model Statistics:
Vocabulary size: 66168 unique words

Unigram model: 1756302 total words counted
Bigram model: 66168 unique contexts, 1756301 total bigrams
Trigram model: 708743 unique contexts, 1756300 total trigrams
Quadgram model: 1374727 unique contexts, 1756299 total quadgrams


## Calculating Perplexity & Laplace Smoothing

To evaluate how well our language model performs, we use a metric called **Perplexity**. Intuitively, perplexity measures how "surprised" the model is by the test data. A lower perplexity means the model is better at predicting the text.

The perplexity is calculated using the exponential of the negative average log probability:
$$Perplexity = \exp\left(-\frac{1}{N} \sum_{i} \ln P(w_i | \text{context})\right)$$

### Handling Unseen Words (Add-One Smoothing)
If our model encounters an N-gram in the test set that it never saw in the training set, the probability would evaluate to zero, which breaks the log calculation (since $\log(0)$ is undefined). To fix this, we apply **Laplace (Add-One) Smoothing**. We add 1 to the count of every N-gram, and add the total vocabulary size ($V$) to the denominator:
$$P(w_i | \text{context}) = \frac{C(\text{n-gram}) + 1}{C(\text{context}) + V}$$

In [36]:
# Function to calculate perplexity on test data
def calculate_perplexity(test_tokens, n, n_grams_count, context_count, vocab_size, total_train_words=0):
    start_index = n - 1  # start from the first n-gram
    N_test = len(test_tokens) - start_index  # number of n-grams in test set
    log_prob_sum = 0.0  # sum of log probabilities
    
    for i in range(start_index, len(test_tokens)):
        if n == 1:  # unigram case
            n_gram = (test_tokens[i],)
            context = ()  # no context for unigram
        else:
            n_gram = tuple(test_tokens[i - n + 1:i + 1])  # current n-gram
            context = tuple(test_tokens[i - n + 1:i])  # context is previous n-1 tokens
        
        count_ngram = n_grams_count.get(n_gram, 0)  # count of current n-gram (0 if unseen)
        
        if n == 1:  # unigram probability
            count_context = total_train_words  # total words in training set
        else:
            count_context = context_count.get(context, 0)  # count of context (0 if unseen)
        
        prob = (count_ngram + 1) / (count_context + vocab_size)  # add-one smoothing
        log_prob_sum += math.log(prob)  # accumulate log probability
    
    perplexity = math.exp(-log_prob_sum / N_test)  # calculate perplexity
    return perplexity

## Evaluating the Models: Calculating Perplexity

Now that our models are trained on the training data, we need to measure how well they perform on our unseen test data (`test_tokens`). In language modeling, the standard evaluation metric is **perplexity**. 

This section executes the evaluation process in three main steps:

* **Extracting Global Statistics**: We retrieve the total vocabulary size (`V`) and the total number of words in our training set. These numbers are essential for the underlying probability calculations and smoothing operations during evaluation.
* **Flattening the N-Gram Counts**: Inside our `NGramModel` class, counts are stored in nested dictionaries (e.g., `context -> target -> count`). To feed these into our evaluation function, we use dictionary comprehensions to "flatten" them. This creates new dictionaries where the key is the complete sequence tuple (e.g., `(word1, word2) -> count`).
* **Calculating & Comparing Scores**: We pass our test data, flattened n-gram counts, context counts, and vocabulary size into the `calculate_perplexity` function. Finally, we print the scores side-by-side. Typically, as the context window increases from Unigram to Trigram, you will see the perplexity score drop, indicating a smarter model.

In [37]:
# Get vocabulary and counts from the trained models
vocabulary = unigram_model.vocab
V = len(vocabulary)  # vocabulary size
total_train_words = len(train_tokens)  # total words in training set

# Extract counts from models
unigrams_count = {(word,): count for word, count in unigram_model.ngram_counts[()].items()}
bigrams_count = {context + (word,): count for context, counter in bigram_model.ngram_counts.items() for word, count in counter.items()}
trigrams_count = {context + (word,): count for context, counter in trigram_model.ngram_counts.items() for word, count in counter.items()}

# Calculate perplexities
uni_ppl = calculate_perplexity(test_tokens, 1, unigrams_count, {}, V, total_train_words)
bigram_ppl = calculate_perplexity(test_tokens, 2, bigrams_count, bigram_model.context_counts, V)
trigram_ppl = calculate_perplexity(test_tokens, 3, trigrams_count, bigram_model.context_counts, V)

print(f"Unigram Perplexity: {uni_ppl}")
print(f"Bigram Perplexity: {bigram_ppl}")
print(f"Trigram Perplexity: {trigram_ppl}")

Unigram Perplexity: 2314.10561144778
Bigram Perplexity: 9592.958448308858
Trigram Perplexity: 41931.552164129826


## Saving the Model

Training an N-gram model on a large dataset like WikiText can take a significant amount of time and memory. Instead of recalculating these frequencies every time we open our notebook, we can save our trained models to our local disk. 

We will use Python's built-in `pickle` module, which serializes (converts) Python objects—like our sets and frequency dictionaries—into byte streams that can be saved as `.pkl` files. 

Our `save_model` function packages the necessary components into a single dictionary and writes it to a file:
* The N-gram size ($n$).
* The vocabulary set (to know our vocabulary size, $V$).
* The N-gram frequency dictionary.
* The Context frequency dictionary (used for the denominator in our probabilities).

In [38]:
# Function to save model parameters
def save_model(n, vocab, n_grams_count, context_count, filename):
    model_parameters = {
        'n': n,  # The N-gram size
        'vocab': vocab,  # Your set of unique words (V)
        'ngram_counts': n_grams_count,  # Your N-gram frequency dictionary
        'context_counts': context_count  # Your Context frequency dictionary
    }
    with open(filename, 'wb') as file:
        pickle.dump(model_parameters, file)
    print(f"Model successfully saved to {filename}!")
    # Save the trained models
save_model(2, vocabulary, bigrams_count, bigram_model.context_counts, 'bigram_model.pkl')
save_model(3, vocabulary, trigrams_count, bigram_model.context_counts, 'trigram_model.pkl')

Model successfully saved to bigram_model.pkl!
Model successfully saved to trigram_model.pkl!


## Generating Text (Greedy Decoding)

Now for the fun part: using our trained models to write new sentences! 



We define a `generate_text` function that performs the following steps:
1. **Load the Model:** It opens our saved `.pkl` file and extracts the vocabulary, N-gram size, and frequency dictionaries.
2. **Contextualize:** It reads our starting prompt and looks at the last $n-1$ words to form the current context.
3. **Predict:** It iterates through the *entire* vocabulary to calculate the smoothed probability of every possible next word.
4. **Select:** It picks the word with the highest probability (a method known as **Greedy Decoding**) and appends it to our sequence.
5. **Repeat:** The context window slides forward, and the process repeats until we hit our `max_tokens` limit or generate an `</end>` tag.

In [39]:
# Function to generate text using a saved model
def generate_text(prompt, model_path, max_tokens=20):
    with open(model_path, 'rb') as file:
        model_parameters = pickle.load(file)
    
    n = model_parameters['n']  # N-gram size
    vocab = model_parameters['vocab']  # Vocabulary set
    n_grams_count = model_parameters['ngram_counts']  # N-gram frequency dictionary
    context_count = model_parameters['context_counts']  # Context frequency dictionary
    V = len(vocab)  # Vocabulary size
    
    tokens = prompt if isinstance(prompt, list) else word_tokenize(prompt.lower())  # tokenize the prompt
    
    for _ in range(max_tokens):
        if n == 1:
            context = ()  # no context for unigram
        else:
            context = tuple(tokens[-(n - 1):])  # get the last n-1 tokens as context
        
        best_word = None
        best_prob = 0.0
        
        for word in vocab:
            current_ngram = context + (word,)  # form the current n-gram (previous context + current word)
            count_ngram = n_grams_count.get(current_ngram, 0)  # count of current n-gram (0 if unseen)
            count_context = context_count.get(context, 0) if n > 1 else sum(context_count.values())  # count of context
            prob = (count_ngram + 1) / (count_context + V)  # add-one smoothing
            
            if prob > best_prob:  # update best word if current probability is higher
                best_prob = prob
                best_word = word
        
        tokens.append(best_word)  # add the best word to the tokens list
        if best_word == '</end>':  # stop if end token is generated
            break
    
    return ' '.join(tokens)  # return the generated text as a string



## Text Generation & Model Comparison

In this section, we put our trained models to the test by having them generate brand new text based on a starting seed phrase. This is a great way to visually evaluate the difference between our models!

* **Setting the Seed Prompt**: We start with the seed phrase `"the united states"` and use `.split()` to break it down into a list of individual tokens, matching the format our model expects.
* **Generating the Sequences**: We pass this tokenized prompt into our `generate_text` function. We load both our serialized Bigram and Trigram models (`.pkl` files) and instruct them to predict a sequence of 20 subsequent words (`max_tokens=20`).



* **Comparing the Results**: Finally, we print the outputs side-by-side. This allows us to directly observe the difference in text quality. Typically, you'll notice that the Trigram model produces more coherent and contextually accurate sentences than the Bigram model, since it relies on a larger window of historical context to make its predictions.

In [42]:
prompt = "the united states"
prompt = prompt.split()
print(prompt)
generated_text_bigram = generate_text(prompt, 'bigram_model.pkl', max_tokens=20)
generated_text_trigram = generate_text(prompt, 'trigram_model.pkl', max_tokens=20)
print(f"Generated text using bigram model: {generated_text_bigram}")
print(f"Generated text using trigram model: {generated_text_trigram}")

['the', 'united', 'states']
Generated text using bigram model: the united states that the first time the first time the first time the first time the first time the first time the
Generated text using trigram model: the united states that the first time the first time the first time the first time the first time the first time the storm s center the congress of mexico the depression was situated over the next day the storm s center the
